# Module 2 — Sentiment / Emotion Classifier

**Goal:** classify a customer message as **negative / neutral / positive** so the system can pick a more apologetic tone for upset customers.

**Dataset:** `dair-ai/emotion` (6 emotions on tweets) → we collapse to 3 buckets:
- **negative** = anger, fear, sadness
- **positive** = joy, love
- **neutral** = surprise

**Model:** fine-tuned DistilBERT (a small Transformer) — this is the "Transformer" option the assignment allows (simpler and more accurate than training an RNN from scratch).

**Where to run this:** you need a GPU for reasonable speed. Use **Google Colab** (free): Runtime → Change runtime type → GPU (T4). Local CPU also works but will be slower (10-20 min).

Run cells top to bottom.

In [ ]:
# 1. Install dependencies
!pip install -q transformers datasets accelerate evaluate scikit-learn

In [ ]:
# 2. Load dataset
from datasets import load_dataset

ds = load_dataset("dair-ai/emotion")
print(ds)
# label meanings in this dataset: 0 sadness, 1 joy, 2 love, 3 anger, 4 fear, 5 surprise
print(ds["train"][0])

In [ ]:
# 3. Map 6 fine-grained emotions -> 3 buckets for routing
# 0 sadness -> negative, 1 joy -> positive, 2 love -> positive,
# 3 anger -> negative, 4 fear -> negative, 5 surprise -> neutral
LABEL_MAP = {0: 0, 3: 0, 4: 0,   # negative
             1: 1, 2: 1,          # positive
             5: 2}                # neutral
ID2NAME = {0: "negative", 1: "positive", 2: "neutral"}

def remap(example):
    example["label3"] = LABEL_MAP[example["label"]]
    return example

ds = ds.map(remap)
print(ds["train"][0])

In [ ]:
# 4. Tokenize
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=64)

ds_tok = ds.map(tokenize, batched=True)
ds_tok = ds_tok.rename_column("label3", "labels")
ds_tok.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

In [ ]:
# 5. Load model + define training
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
import evaluate

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1_macro": f1.compute(predictions=preds, references=labels, average="macro")["f1"],
    }

args = TrainingArguments(
    output_dir="./sentiment_ckpt",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=2,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_tok["train"],
    eval_dataset=ds_tok["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

In [ ]:
# 6. Train (this is the slow step — a few minutes on a Colab GPU)
trainer.train()

In [ ]:
# 7. Evaluate on the held-out test split
metrics = trainer.evaluate(ds_tok["test"])
print(metrics)

In [ ]:
# 8. Quick sanity check
import torch

def predict_sentiment(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=64).to(model.device)
    with torch.no_grad():
        logits = model(**inputs).logits
    pred_id = int(torch.argmax(logits, dim=-1)[0])
    return ID2NAME[pred_id]

for t in [
    "This is the third time my order has been late and nobody replies to me!",
    "Thanks so much, my package arrived earlier than expected!",
    "Can you tell me the delivery time for order #4521?",
]:
    print(predict_sentiment(t), "->", t)

In [ ]:
# 9. Save the fine-tuned model for deployment
import os
os.makedirs("../models/sentiment_model", exist_ok=True)
model.save_pretrained("../models/sentiment_model")
tokenizer.save_pretrained("../models/sentiment_model")
print("Saved to ../models/sentiment_model")